# Cabardo Studio AI Generator FREE Lifetime — 1-Click Free Colab T4 16 GB Backend

**Zero API keys · Zero cost · Just click `Runtime -> Run all`:**
1. Confirm top-right says **T4** (if not: `Runtime -> Change runtime type -> T4 GPU -> Save`).
2. Click **Runtime → Run all** (takes ~45 seconds).
3. Copy the printed `https://*.trycloudflare.com` URL into **https://flow-studio-pwa.vercel.app → Engine Settings → Test Connection**.


In [ ]:
# @title Step 1: Verify Free NVIDIA T4 GPU & Install Dependencies (~35s, once per session)
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU attached! Click Runtime -> Change runtime type -> T4 GPU -> Save, then click Run All."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"GPU Ready: {gpu_name} ({vram_gb:.1f} GB VRAM)")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
!pip -q install -U diffusers transformers accelerate safetensors sentencepiece protobuf ftfy imageio[ffmpeg] pillow fastapi uvicorn python-multipart
print("All Cabardo Studio T4 dependencies installed cleanly.")


In [ ]:
# @title Step 2: Start Cabardo Studio AI Generator FREE Lifetime Backend Server (FastAPI + Free Cloudflare Tunnel)
import asyncio
import base64
import gc
import io
import os
import re
import subprocess
import threading
import time
import uuid
import urllib.request
from typing import Optional

import torch
from PIL import Image
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from pydantic import BaseModel
import uvicorn

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU attached! Click Runtime -> Change runtime type -> T4 GPU -> Save, then Run All again."
    )

os.makedirs("outputs", exist_ok=True)

app = FastAPI(title="Cabardo Studio Colab T4 Backend")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)

GPU_LOCK = threading.Lock()
STATE = {
    "active_pipe_key": None,
    "pipe": None,
    "jobs": {},
}

def _unload_pipe():
    if STATE["pipe"] is not None:
        del STATE["pipe"]
        STATE["pipe"] = None
        STATE["active_pipe_key"] = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def _load_image_pipe(model_key: str):
    target_key = f"img:{model_key}"
    if STATE["active_pipe_key"] == target_key and STATE["pipe"] is not None:
        return STATE["pipe"]
    _unload_pipe()
    from diffusers import AutoPipelineForText2Image
    if model_key == "sdxl-turbo":
        pipe = AutoPipelineForText2Image.from_pretrained(
            "stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16"
        ).to("cuda")
    else:
        pipe = AutoPipelineForText2Image.from_pretrained(
            "stabilityai/stable-diffusion-xl-base-1.0",
            torch_dtype=torch.float16,
            variant="fp16",
            use_safetensors=True,
        )
        pipe.enable_model_cpu_offload()
        pipe.enable_vae_slicing()
        pipe.enable_vae_tiling()
    STATE["pipe"] = pipe
    STATE["active_pipe_key"] = target_key
    return pipe

def _load_video_pipe(model_key: str, is_i2v: bool):
    target_key = f"vid:{model_key}:{'i2v' if is_i2v else 't2v'}"
    if STATE["active_pipe_key"] == target_key and STATE["pipe"] is not None:
        return STATE["pipe"]
    _unload_pipe()
    if model_key == "ltx-video":
        if is_i2v:
            from diffusers import LTXImageToVideoPipeline
            pipe = LTXImageToVideoPipeline.from_pretrained(
                "Lightricks/LTX-Video", torch_dtype=torch.float16
            )
        else:
            from diffusers import LTXPipeline
            pipe = LTXPipeline.from_pretrained(
                "Lightricks/LTX-Video", torch_dtype=torch.float16
            )
        pipe.enable_sequential_cpu_offload()
        pipe.vae.enable_tiling()
    elif model_key == "wan2.1-1.3b":
        from diffusers import WanPipeline
        pipe = WanPipeline.from_pretrained(
            "Wan-AI/Wan2.1-T2V-1.3B-Diffusers", torch_dtype=torch.float16
        )
        pipe.enable_sequential_cpu_offload()
    else:
        from diffusers import CogVideoXPipeline
        pipe = CogVideoXPipeline.from_pretrained(
            "THUDM/CogVideoX-2b", torch_dtype=torch.float16
        )
        pipe.enable_sequential_cpu_offload()
        pipe.vae.enable_slicing()
        pipe.vae.enable_tiling()
    STATE["pipe"] = pipe
    STATE["active_pipe_key"] = target_key
    return pipe

class ImageRequest(BaseModel):
    prompt: str
    negative_prompt: str = "blurry, low quality, distorted, watermark"
    width: int = 1024
    height: int = 576
    steps: int = 20
    cfg_scale: float = 7.0
    seed: int = -1
    model: str = "sdxl-turbo"

class VideoRequest(BaseModel):
    prompt: str
    negative_prompt: str = "blurry, jittery, distorted, worst quality"
    width: int = 720
    height: int = 480
    duration_sec: int = 4
    steps: int = 25
    cfg_scale: float = 6.0
    seed: int = -1
    model: str = "cogvideox-2b"
    mode: str = "t2v"
    start_frame_data_url: Optional[str] = None

@app.get("/api/health")
def health():
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    total_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 1) if torch.cuda.is_available() else 0.0
    free_gb = round(torch.cuda.mem_get_info(0)[0] / (1024 ** 3), 1) if torch.cuda.is_available() else 0.0
    return {
        "ok": True,
        "gpu_name": gpu_name,
        "vram_total_gb": total_gb,
        "vram_free_gb": free_gb,
        "active_pipeline": STATE["active_pipe_key"],
        "models": ["sdxl-turbo", "sdxl-base", "cogvideox-2b", "ltx-video", "wan2.1-1.3b"],
    }

@app.post("/api/image")
def generate_image(req: ImageRequest):
    actual_seed = int(time.time() * 1000) % (2**31) if req.seed < 0 else int(req.seed)
    with GPU_LOCK:
        pipe = _load_image_pipe(req.model)
        is_turbo = req.model == "sdxl-turbo"
        w = min(max(256, (req.width // 16) * 16), 768 if is_turbo else 1152)
        h = min(max(256, (req.height // 16) * 16), 768 if is_turbo else 1152)
        steps = min(max(1, req.steps), 4) if is_turbo else min(max(12, req.steps), 40)
        cfg = 0.0 if is_turbo else float(req.cfg_scale)
        gen = torch.Generator(device="cuda").manual_seed(actual_seed)
        img = pipe(
            prompt=req.prompt,
            negative_prompt=None if is_turbo else req.negative_prompt,
            width=w,
            height=h,
            num_inference_steps=steps,
            guidance_scale=cfg,
            generator=gen,
        ).images[0]
        fname = f"img_{int(time.time())}_{actual_seed}.png"
        fpath = os.path.join("outputs", fname)
        img.save(fpath)
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        b64 = base64.b64encode(buf.getvalue()).decode("ascii")
        return {
            "ok": True,
            "seed": actual_seed,
            "width": w,
            "height": h,
            "output_path": f"/outputs/{fname}",
            "data_url": f"data:image/png;base64,{b64}",
        }

def _run_video_worker(job_id: str, req: VideoRequest):
    from diffusers.utils import export_to_video
    job = STATE["jobs"][job_id]
    try:
        actual_seed = int(time.time() * 1000) % (2**31) if req.seed < 0 else int(req.seed)
        is_i2v = bool(req.mode == "i2v" and req.start_frame_data_url and req.start_frame_data_url.startswith("data:image/"))
        model_key = req.model
        if is_i2v and model_key == "cogvideox-2b":
            model_key = "ltx-video"

        job["status"] = "processing"
        job["progress"] = 15
        job["phase"] = f"Loading {model_key} on NVIDIA T4 (fp16)..."

        with GPU_LOCK:
            pipe = _load_video_pipe(model_key, is_i2v)
            job["progress"] = 35
            job["phase"] = f"Diffusing video frames ({model_key})..."
            gen = torch.Generator(device="cuda").manual_seed(actual_seed)

            if model_key == "ltx-video":
                w = max(384, min(704, (req.width // 32) * 32))
                h = max(384, min(512, (req.height // 32) * 32))
                num_frames = min(65, max(17, int(req.duration_sec * 16) + 1))
                num_frames = ((num_frames - 1) // 8) * 8 + 1
                kwargs = dict(
                    prompt=req.prompt,
                    negative_prompt=req.negative_prompt,
                    width=w,
                    height=h,
                    num_frames=num_frames,
                    num_inference_steps=min(max(12, req.steps), 40),
                    guidance_scale=max(2.0, min(7.5, req.cfg_scale)),
                    generator=gen,
                )
                if is_i2v and req.start_frame_data_url:
                    raw_b64 = req.start_frame_data_url.split(",", 1)[1]
                    pil_img = Image.open(io.BytesIO(base64.b64decode(raw_b64))).convert("RGB").resize((w, h))
                    kwargs["image"] = pil_img
                frames = pipe(**kwargs).frames[0]
                fps = 16
            elif model_key == "wan2.1-1.3b":
                frames = pipe(
                    prompt=req.prompt,
                    negative_prompt=req.negative_prompt,
                    height=480,
                    width=832,
                    num_frames=min(49, max(17, int(req.duration_sec * 12) + 1)),
                    num_inference_steps=min(max(15, req.steps), 40),
                    guidance_scale=5.0,
                    generator=gen,
                ).frames[0]
                fps = 15
            else:
                num_frames = min(49, max(17, int(req.duration_sec * 8) + 1))
                frames = pipe(
                    prompt=req.prompt,
                    num_videos_per_prompt=1,
                    num_inference_steps=min(max(15, req.steps), 40),
                    num_frames=num_frames,
                    guidance_scale=float(req.cfg_scale),
                    generator=gen,
                ).frames[0]
                fps = 8

            job["progress"] = 88
            job["phase"] = "Encoding MP4 clip..."
            fname = f"vid_{job_id}_{actual_seed}.mp4"
            fpath = os.path.join("outputs", fname)
            export_to_video(frames, fpath, fps=fps)

            job["status"] = "completed"
            job["progress"] = 100
            job["phase"] = "Completed on Colab T4"
            job["output_path"] = f"/outputs/{fname}"
    except Exception as exc:
        job["status"] = "failed"
        job["error"] = str(exc)

@app.post("/api/video")
def queue_video(req: VideoRequest):
    job_id = f"job_{uuid.uuid4().hex[:12]}"
    STATE["jobs"][job_id] = {
        "job_id": job_id,
        "status": "queued",
        "progress": 5,
        "phase": "Queued for NVIDIA T4 GPU...",
        "output_path": None,
        "error": None,
    }
    threading.Thread(target=_run_video_worker, args=(job_id, req), daemon=True).start()
    return {"ok": True, "job_id": job_id, "status": "queued"}

@app.get("/api/job/{job_id}")
def get_job(job_id: str):
    if not re.fullmatch(r"[A-Za-z0-9_-]{4,64}", job_id) or job_id not in STATE["jobs"]:
        raise HTTPException(status_code=404, detail="Job not found")
    return STATE["jobs"][job_id]

@app.get("/outputs/{filename}")
def get_output(filename: str):
    if not re.fullmatch(r"[A-Za-z0-9._-]{3,128}", filename):
        raise HTTPException(status_code=400, detail="Invalid filename")
    fpath = os.path.join("outputs", filename)
    if not os.path.isfile(fpath):
        raise HTTPException(status_code=404, detail="Output file not found")
    return FileResponse(fpath)

# Start Uvicorn in a clean dedicated asyncio event loop thread (safe inside Colab/IPython)
def _start_api():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    config = uvicorn.Config(app, host="127.0.0.1", port=7860, log_level="warning")
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

if not globals().get("_FLOWSTUDIO_SERVER_STARTED"):
    threading.Thread(target=_start_api, daemon=True).start()
    _FLOWSTUDIO_SERVER_STARTED = True

# Wait until local FastAPI actually responds HTTP 200 before opening Cloudflare tunnel
api_ready = False
for _ in range(30):
    try:
        with urllib.request.urlopen("http://127.0.0.1:7860/api/health", timeout=2) as r:
            if r.status == 200:
                print("Local FastAPI health OK:", r.read().decode("utf-8"))
                api_ready = True
                break
    except Exception:
        time.sleep(0.5)

if not api_ready:
    raise RuntimeError("Local FastAPI server failed to start on port 7860.")

cf_bin = "/usr/local/bin/cloudflared"
if not os.path.exists(cf_bin):
    print("Downloading official cloudflared binary...")
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        cf_bin,
    )
    os.chmod(cf_bin, 0o755)

print("Starting free Cloudflare Quick Tunnel to your T4 GPU...")
proc = subprocess.Popen(
    [cf_bin, "tunnel", "--url", "http://127.0.0.1:7860", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

tunnel_url = None
recent_lines = []
for line in proc.stdout:
    recent_lines.append(line.strip())
    if len(recent_lines) > 20:
        recent_lines.pop(0)
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m and not tunnel_url:
        tunnel_url = m.group(0)
        print("=" * 72)
        print("CABARDO STUDIO AI GENERATOR (T4 GPU) IS LIVE!")
        print("Paste this URL into https://flow-studio-pwa.vercel.app -> Engine Settings:")
        print("\n  " + tunnel_url + "\n")
        print("Keep this cell running while generating from Cabardo Studio AI Generator FREE Lifetime.")
        print("=" * 72)

if not tunnel_url:
    raise RuntimeError("Cloudflare tunnel exited without printing a URL:\n" + "\n".join(recent_lines))
